# Vibe Coder - Colab Model Server

This notebook sets up an OpenAI-compatible API server that connects to your Vibe Coder IDE via Cloudflared tunnel.

## Setup Steps:
1. Run all cells in order
2. Copy the Cloudflared URL to your `.env.local` file as `COLAB_CLOUDFLARED_URL`
3. Start coding in your Vibe Coder IDE!

In [ ]:
# Step 1: Install required packages
!pip install flask flask-cors transformers torch accelerate bitsandbytes openai fastapi uvicorn pyngrok -q

In [ ]:
# Step 2: Download and install cloudflared
import subprocess
import os

# Download cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Step 3: Choose and load your model
# You can use any HuggingFace model. Popular choices for coding:
# - Qwen/Qwen2.5-Coder-7B-Instruct
# - meta-llama/Llama-3.1-8B-Instruct
# - microsoft/Phi-3-mini-4k-instruct

from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

# Model selection - change this to your preferred model
MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"

print(f"Loading model: {MODEL_NAME}...")
print("This may take a few minutes...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Load model with quantization for memory efficiency
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully!")

In [ ]:
# Step 4: Create the OpenAI-compatible API server
%%writefile server.py

from flask import Flask, request, jsonify, Response
from flask_cors import CORS
import json
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import threading
import queue

app = Flask(__name__)
CORS(app)

# Global model reference (set by main notebook)
model = None
tokenizer = None
device = "cuda" if torch.cuda.is_available() else "cpu"

@app.route('/v1/chat/completions', methods=['POST'])
def chat_completions():
    global model, tokenizer
    
    data = request.json
    messages = data.get('messages', [])
    stream = data.get('stream', False)
    max_tokens = data.get('max_tokens', 4096)
    temperature = data.get('temperature', 0.7)
    
    # Convert messages to prompt
    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(chat_text, return_tensors="pt").to(device)
    
    if stream:
        return stream_response(inputs, max_tokens, temperature)
    else:
        return non_stream_response(inputs, max_tokens, temperature)

def non_stream_response(inputs, max_tokens, temperature):
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    
    response = {
        "id": f"chatcmpl-{int(time.time())}",
        "object": "chat.completion",
        "created": int(time.time()),
        "model": "colab-model",
        "choices": [{
            "index": 0,
            "message": {
                "role": "assistant",
                "content": generated_text
            },
            "finish_reason": "stop"
        }],
        "usage": {
            "prompt_tokens": inputs['input_ids'].shape[1],
            "completion_tokens": len(outputs[0]) - inputs['input_ids'].shape[1],
            "total_tokens": len(outputs[0])
        }
    }
    
    return jsonify(response)

def stream_response(inputs, max_tokens, temperature):
    def generate():
        response_id = f"chatcmpl-{int(time.time())}"
        
        # Send initial chunk
        initial_data = {
            "id": response_id,
            "object": "chat.completion.chunk",
            "created": int(time.time()),
            "model": "colab-model",
            "choices": [{
                "index": 0,
                "delta": {"role": "assistant"},
                "finish_reason": None
            }]
        }
        yield f"data: {json.dumps(initial_data)}\n\n"
        
        # Generate tokens one by one for streaming
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                temperature=temperature,
                do_sample=temperature > 0,
                pad_token_id=tokenizer.eos_token_id
            )
        
        generated_ids = outputs[0][inputs['input_ids'].shape[1]:]
        
        # Stream tokens in chunks for better UX
        chunk_size = 5
        for i in range(0, len(generated_ids), chunk_size):
            token_chunk = generated_ids[i:i+chunk_size]
            text = tokenizer.decode(token_chunk, skip_special_tokens=True)
            
            if text:
                chunk_data = {
                    "id": response_id,
                    "object": "chat.completion.chunk",
                    "created": int(time.time()),
                    "model": "colab-model",
                    "choices": [{
                        "index": 0,
                        "delta": {"content": text},
                        "finish_reason": None
                    }]
                }
                yield f"data: {json.dumps(chunk_data)}\n\n"
        
        # Send final chunk
        final_data = {
            "id": response_id,
            "object": "chat.completion.chunk",
            "created": int(time.time()),
            "model": "colab-model",
            "choices": [{
                "index": 0,
                "delta": {},
                "finish_reason": "stop"
            }]
        }
        yield f"data: {json.dumps(final_data)}\n\n"
        yield "data: [DONE]\n\n"
    
    return Response(generate(), mimetype='text/event-stream')

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ok", "model": "loaded" if model is not None else "not loaded"})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)

In [ ]:
# Step 5: Set the global model variables in the server module
import server
server.model = model
server.tokenizer = tokenizer
print("Model connected to server!")

In [ ]:
# Step 6: Start Cloudflared tunnel and Flask server
import threading
import time
import subprocess
import re

# Start Flask server in a thread
def run_flask():
    import os
    os.system('python server.py')

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
print("Flask server starting on port 5000...")
time.sleep(3)  # Wait for Flask to start

# Start cloudflared tunnel
print("Starting Cloudflared tunnel...")
cloudflared_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

# Wait for and extract the URL
cloudflared_url = None
for line in cloudflared_proc.stdout:
    print(line.strip())
    if 'https://' in line and 'trycloudflare.com' in line:
        match = re.search(r'(https://[a-z0-9-]+\.trycloudflare\.com)', line)
        if match:
            cloudflared_url = match.group(1)
            break

if cloudflared_url:
    print(f"\n{'='*60}")
    print(f"Your Cloudflared URL:")
    print(f"{cloudflared_url}")
    print(f"{'='*60}")
    print(f"\nCopy this URL to your .env.local file:")
    print(f"COLAB_CLOUDFLARED_URL={cloudflared_url}")
    print(f"\nThen restart your Next.js app.")
else:
    print("Could not extract cloudflared URL. Check the output above.")

## Server is now running!

Copy the Cloudflared URL from above and paste it into your `.env.local` file:

```
COLAB_CLOUDFLARED_URL=your-https-url.trycloudflare.com
```

Then restart your Next.js development server. Your Vibe Coder IDE should now be connected to the AI model running in Colab!